# Week 63, Day 3: Mean-field variational inference

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: CAVI for a mean and a precision

Complete the CAVI loop: update q(mu) from E[tau], then q(tau) from the mean and variance of q(mu). Compare the results with the exact normal-gamma posterior.

In [ ]:
import numpy as np
g0 = np.random.default_rng(633)
xs = g0.normal(12.0, 3.0, 10)                          # 10 trials of a neuron's response (spikes per second)
n, xbar = len(xs), xs.mean()
mu0, lam0, a0, b0 = 10.0, 0.1, 1.0, 1.0

E_tau = 1.0
for it in range(100):
    m_q = (lam0 * mu0 + n * xbar) / (lam0 + n)
    v_q = ...                                          # variance of q(mu)
    a_q = a0 + (n + 1) / 2
    E_sq = np.sum((xs - m_q) ** 2) + n * v_q + lam0 * ((m_q - mu0) ** 2 + v_q)
    b_q = b0 + 0.5 * E_sq
    E_tau = ...

# exact normal-gamma posterior
lam_n = lam0 + n; m_exact = (lam0 * mu0 + n * xbar) / lam_n
a_n = a0 + n / 2; b_n = b0 + 0.5 * np.sum((xs - xbar) ** 2) + lam0 * n * (xbar - mu0) ** 2 / (2 * lam_n)
Etau_exact = a_n / b_n
var_mu_exact = b_n / (a_n - 1) / lam_n                # marginal of mu is a Student t
print(f"mean of mu: variational {m_q:.3f}, exact {m_exact:.3f}")
print(f"mean of tau: variational {E_tau:.4f}, exact {Etau_exact:.4f}  (noise sd about {1 / np.sqrt(E_tau):.2f})")
print(f"variance of mu: variational {v_q:.4f}, exact {var_mu_exact:.4f}")

In [ ]:
import numpy as np
def test_mean():
    "the variational mean of mu matches the exact posterior mean"
    assert abs(m_q - m_exact) < 1e-6
def test_tau():
    "the variational mean of tau is within 5% of the exact posterior mean"
    assert abs(E_tau / Etau_exact - 1) < 0.05

test_mean()
test_tau()
print("All checks passed")

### Exercise: Mean field against enumeration

For 8 units with random couplings, iterate the mean-field equations (damped: average the new and old values) to convergence, and compare with exact means from enumerating all 256 states. Do this for couplings of scale 0.1 and 0.4.

In [ ]:
import itertools
S8 = np.array(list(itertools.product((-1, 1), repeat=8)))
gi = np.random.default_rng(6331)
h8 = gi.normal(0, 0.3, 8)
Jbase = np.triu(gi.normal(0, 1, (8, 8)), 1); Jbase = Jbase + Jbase.T

def exact_means(J):
    E = S8 @ h8 + 0.5 * np.einsum("ki,ij,kj->k", S8, J, S8)
    p = np.exp(E - E.max()); p /= p.sum()
    return S8.T @ p

def mean_field(J, iters=500):
    m = np.zeros(8)
    for _ in range(iters):
        m = 0.5 * m + 0.5 * ...
    return m

err = {}
for scale in (0.1, 0.4):
    J = scale * Jbase
    err[scale] = np.abs(mean_field(J) - exact_means(J)).max()
    print(f"coupling scale {scale}: exact {np.round(exact_means(J), 3)}")
    print(f"                   mean field {np.round(mean_field(J), 3)}  largest error {err[scale]:.4f}")
err_weak, err_strong = err[0.1], err[0.4]

In [ ]:
import numpy as np
def test_weak():
    "with weak couplings, mean-field means are within 0.05 of exact"
    assert err_weak < 0.05
def test_strong():
    "with couplings four times stronger, the error is several times larger"
    assert err_strong > 3 * err_weak

test_weak()
test_strong()
print("All checks passed")

### Exercise: Variance of a mean-field Gaussian

For a two-dimensional Gaussian posterior with unit variances and correlation rho, run CAVI (each factor's mean and variance from the conditional given the other's mean) and compare the mean-field variance with the true one, for rho = 0, 0.5 and 0.9.

In [ ]:
ratio = {}
for rho in (0.0, 0.5, 0.9):
    Sigma = np.array([[1, rho], [rho, 1]]); Lam = np.linalg.inv(Sigma); mu_t = np.array([1.0, -1.0])
    m = np.zeros(2)
    for _ in range(200):
        for j in (0, 1):
            o = 1 - j
            m[j] = mu_t[j] - Lam[j, o] / Lam[j, j] * (m[o] - mu_t[o])
    v_mf = ...
    ratio[rho] = v_mf[0] / Sigma[0, 0]
    print(f"rho {rho}: mean-field means {np.round(m, 3)}, variance {v_mf[0]:.3f} against true {Sigma[0, 0]:.3f}")

In [ ]:
import numpy as np
def test_ratio():
    "for correlation 0.9 the mean-field variance is about 0.19 of the true variance; for zero correlation it is exact"
    assert abs(ratio[0.9] - 0.19) < 0.01 and abs(ratio[0.0] - 1) < 1e-9

test_ratio()
print("All checks passed")